In [2]:
!pip install transformers datasets torch scikit-learn mlflow --break-system-packages

  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached mlflow-3.14.0-py3-none-any.whl.metadata (49 kB)
  Using cached huggingface_hub-1.20.1-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typer-0.26.7-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mlflow_skinny-3.14.0-p

In [1]:
import torch
import transformers
import mlflow
print(torch.__version__, transformers.__version__, mlflow.__version__)
print(torch.backends.mps.is_available())

2.12.1 5.12.1 3.14.0
True


In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# Veriyi yükle
df = pd.read_csv('/Users/erenbey/amazon-fine-food-reviews/balanced_reviews.csv')

# Label encoding
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

# Text alanını hazırla
df['text'] = df['Text'].astype(str)

# Train/val/test split (%80/%10/%10)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print('Train:', len(train_df))
print('Val:', len(val_df))
print('Test:', len(test_df))

train_df.head(3)

Train: 96000
Val: 12000
Test: 12000


,Id,ProductId,Time,Score,sentiment,Summary,Text,label,text
80060,206299,B003CGJGK4,1340064000,4,positive,Doesn't fit in bags-on-board style dog bag dis...,My singular complaint is that these bags don't...,2,My singular complaint is that these bags don't...
60528,43494,B001EQ4P2I,1297036800,5,positive,Best Nuts Ever,I absolutely love these nuts! I can't find the...,2,I absolutely love these nuts! I can't find the...
112154,260135,B000NMJWZO,1323302400,5,positive,life saver!,We love Pamelas products in our home. They ha...,2,We love Pamelas products in our home. They ha...


In [3]:
from transformers import DistilBertTokenizerFast
import torch
from torch.utils.data import Dataset

# Tokenizer yükle
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Custom Dataset class
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Dataset objelerini oluştur
train_dataset = SentimentDataset(train_df['text'], train_df['label'], tokenizer)
val_dataset = SentimentDataset(val_df['text'], val_df['label'], tokenizer)
test_dataset = SentimentDataset(test_df['text'], test_df['label'], tokenizer)

print('Train dataset size:', len(train_dataset))

# Bir örneğe bakalım
sample = train_dataset[0]
print('Input IDs shape:', sample['input_ids'].shape)
print('Attention mask shape:', sample['attention_mask'].shape)
print('Label:', sample['labels'])

Train dataset size: 96000
Input IDs shape: torch.Size([256])
Attention mask shape: torch.Size([256])
Label: tensor(2)


In [7]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

# Model yükle (3 sınıf: negative, neutral, positive)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3
)

# Cihazı ayarla (M1 Pro için MPS)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
print('Using device:', device)

# Metrik hesaplama fonksiyonu
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Eğitim ayarları
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none'  # MLflow'u manuel entegre edeceğiz
)

# Trainer oluştur
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print('Trainer hazır, eğitime başlanabilir.')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [4]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

# Model yükle (3 sınıf: negative, neutral, positive)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3
)

# Cihazı ayarla (M1 Pro için MPS)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
print('Using device:', device)

# Metrik hesaplama fonksiyonu
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Eğitim ayarları
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none'  # MLflow'u manuel entegre edeceğiz
)

# Trainer oluştur
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print('Trainer hazır, eğitime başlanabilir.')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Using device: mps
Trainer hazır, eğitime başlanabilir.


In [9]:
!pip install accelerate --break-system-packages


In [5]:
trainer.train()

/opt/anaconda3/envs/tf-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [6]:
sample_lengths = train_df['text'].str[:500].apply(lambda x: len(tokenizer.encode(str(x))))
print('Mean token length:', sample_lengths.mean())
print('Median:', sample_lengths.median())
print('95th percentile:', sample_lengths.quantile(0.95))
print('Max:', sample_lengths.max())

Mean token length: 82.82805208333333
Median: 82.0
95th percentile: 134.0
Max: 238


In [7]:
# Dataset'leri yeni max_length ile yeniden oluştur
train_dataset = SentimentDataset(train_df['text'], train_df['label'], tokenizer, max_length=128)
val_dataset = SentimentDataset(val_df['text'], val_df['label'], tokenizer, max_length=128)
test_dataset = SentimentDataset(test_df['text'], test_df['label'], tokenizer, max_length=128)

# Modeli sıfırdan yükle (önceki kısmi eğitim ağırlıklarını temizlemek için)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3
)
model.to(device)

# Trainer'ı yeniden oluştur
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print('Dataset ve Trainer 128 token ile yeniden hazırlandı.')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dataset ve Trainer 128 token ile yeniden hazırlandı.


In [8]:
trainer.train()

/opt/anaconda3/envs/tf-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.494396,0.498220,0.800250,0.799744,0.800037,0.800250
2,0.391147,0.461542,0.826583,0.825652,0.826252,0.826583
3,0.178271,0.583846,0.832667,0.833266,0.834155,0.832667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/envs/tf-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/envs/tf-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18000, training_loss=0.3967590691248576, metrics={'train_runtime': 8431.658, 'train_samples_per_second': 34.157, 'train_steps_per_second': 2.135, 'total_flos': 9537822793728000.0, 'train_loss': 0.3967590691248576, 'epoch': 3.0})

In [9]:
trainer.save_model('./final_model')
tokenizer.save_pretrained('./final_model')
print('Model ve tokenizer kaydedildi.')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model ve tokenizer kaydedildi.


In [10]:
from torch.utils.data import DataLoader

test_loader = DataLoader(test_dataset, batch_size=32)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print('Tahmin tamamlandı, toplam:', len(all_preds))

Tahmin tamamlandı, toplam: 12000


In [12]:
from sklearn.metrics import classification_report, confusion_matrix

In [13]:
label_names = ['negative', 'neutral', 'positive']

print(classification_report(all_labels, all_preds, target_names=label_names))
print()
print('Confusion Matrix:')
print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

    negative       0.85      0.84      0.84      4000
     neutral       0.77      0.79      0.78      4000
    positive       0.90      0.87      0.88      4000

    accuracy                           0.84     12000
   macro avg       0.84      0.84      0.84     12000
weighted avg       0.84      0.84      0.84     12000


Confusion Matrix:
[[3371  533   96]
 [ 525 3162  313]
 [  88  419 3493]]


In [14]:
import pandas as pd

# Orijinal tüm veriyi yükle (568K satır)
full_df = pd.read_csv('/Users/erenbey/amazon-fine-food-reviews/Reviews.csv')
full_df['date'] = pd.to_datetime(full_df['Time'], unit='s')

print('Toplam satır:', len(full_df))
print('Tarih aralığı:', full_df['date'].min(), '-', full_df['date'].max())

Toplam satır: 568454
Tarih aralığı: 1999-10-08 00:00:00 - 2012-10-26 00:00:00


In [15]:
import time
from torch.utils.data import DataLoader, Dataset

class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }

# Hız testi: ilk 2000 satır ile
test_sample = full_df['Text'].head(2000)
test_inf_dataset = InferenceDataset(test_sample, tokenizer)
test_inf_loader = DataLoader(test_inf_dataset, batch_size=32)

model.eval()
start = time.time()
with torch.no_grad():
    for batch in test_inf_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
elapsed = time.time() - start

print(f'2000 satır için süre: {elapsed:.1f} saniye')
print(f'568454 satır için tahmini süre: {elapsed * 568454/2000 / 60:.1f} dakika')

2000 satır için süre: 11.7 saniye
568454 satır için tahmini süre: 55.6 dakika


In [16]:
test_inf_loader = DataLoader(test_inf_dataset, batch_size=64)
# aynı hız testini tekrar çalıştır

In [17]:
import time

model.eval()
start = time.time()
with torch.no_grad():
    for batch in test_inf_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
elapsed = time.time() - start

print(f'2000 satır için süre (batch_size=64): {elapsed:.1f} saniye')
print(f'568454 satır için tahmini süre: {elapsed * 568454/2000 / 60:.1f} dakika')

2000 satır için süre (batch_size=64): 14.3 saniye
568454 satır için tahmini süre: 67.7 dakika


In [ ]:
import time

# Tüm veri için dataset oluştur
full_inf_dataset = InferenceDataset(full_df['Text'], tokenizer)
full_inf_loader = DataLoader(full_inf_dataset, batch_size=32)

model.eval()
all_predictions = []

start = time.time()
with torch.no_grad():
    for i, batch in enumerate(full_inf_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_predictions.extend(preds)
        
        # İlerleme göstergesi - her 1000 batch'te bir
        if i % 1000 == 0:
            elapsed = time.time() - start
            print(f'Batch {i}, geçen süre: {elapsed/60:.1f} dakika, işlenen: {i*32}/{len(full_df)}')

total_elapsed = time.time() - start
print(f'\nTamamlandı! Toplam süre: {total_elapsed/60:.1f} dakika')
print(f'Toplam tahmin: {len(all_predictions)}')

Batch 0, geçen süre: 0.0 dakika, işlenen: 0/568454
